# Semana 9: RQ4, produtos do EMSR720 de 06/05/2024

Seção 2 de `prereg/plano_rq4.md`, entrada de 2026-11-02 sobre o EMSR720. **Este notebook não roda nenhum modelo e não lê nenhuma saída de modelo.** Ele baixa os cinco produtos de mapeamento de danos (GRA_PRODUCT) do Copernicus EMS feitos com imagens Pléiades Neo de 06/05/2024 (AOI 1 a 5: Guaporé, Encantado, Roca Sales, Barragem das Antas, Santa Tereza) e faz o inventário das camadas vetoriais: número de feições, classes (área inundada ou vestígio de inundação), e área dentro de cada um dos três tiles. Nada é rasterizado nem comparado com nada ainda.

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive, clona o repositório, confere os três tiles |
| 2 | Escolhe os produtos pela regra do plano (GRA_PRODUCT, todas as imagens de 06/05/2024) e mostra a tabela. Não baixa nada |
| 3 | Baixa os cinco pacotes (`.zip`), confere a integridade e extrai tudo menos as imagens |
| 4 | Inventário de todas as camadas vetoriais e das classes |
| 5 | Copia as camadas `observedEvent` para a pasta de saída e grava o registro da rodada |

Abra o Colab logada na conta do mestrado (não precisa de GPU). Se alguma célula der erro, pare e me mande a saída inteira. Os pacotes baixados ficam em `dados/rs2024/referencias/emsr720` e não são baixados de novo numa segunda rodada. A saída de cada rodada vai para uma pasta nova (`rq4_emsr720`, depois `rq4_emsr720_2`, e assim por diante); a célula 1 mostra qual.

Depois da célula 5, me mande **todos os arquivos** da pasta de saída desta rodada (são pequenos).

In [ ]:
# Célula 1 de 5: Drive e repositório
import os, re, json, time, hashlib, glob, shutil, subprocess, sys, zipfile
import requests

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

REF = f'{RAIZ}/dados/rs2024/referencias'
SAIDA_BASE = f'{RAIZ}/resultados/semana9/rq4_emsr720'
LOCAL = '/content/emsr720'
# Nunca sobrescreve: se a pasta já tem arquivos de uma rodada anterior, usa a próxima livre (_2, _3, ...).
SAIDA, n = SAIDA_BASE, 1
while os.path.isdir(SAIDA) and os.listdir(SAIDA):
    n += 1
    SAIDA = f'{SAIDA_BASE}_{n}'
print('Saída desta rodada:', SAIDA.replace(MEUDRIVE, 'MyDrive'))
for d in (REF, SAIDA, LOCAL):
    os.makedirs(d, exist_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyogrio'], check=True)

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO_GIT}/analysis/rq4')
import inventario_referencias as I
CAIXAS = I.caixas_tiles(f'{REPO_GIT}/analysis/rq4/entradas/rq4_entradas.json')
assert set(CAIXAS) == {'22JDM', '22JCN', '22JDN'}, f'O registro das entradas deveria ter os três tiles, tem {sorted(CAIXAS)}. Pare e me mande esta saída.'

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

def zip_integro(caminho):
    try:
        with zipfile.ZipFile(caminho) as z:
            return z.testzip() is None
    except Exception:
        return False

def baixar_zip(url, destino):
    # Usa o pacote do Drive se ele existir e estiver íntegro; senão baixa para o disco local e copia para o Drive.
    no_drive = f'{REF}/{os.path.relpath(destino, LOCAL)}'
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    if os.path.exists(no_drive) and not os.path.exists(destino):
        shutil.copyfile(no_drive, destino)
    if os.path.exists(destino) and zip_integro(destino):
        return destino
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(destino + '.parcial', 'wb') as f:
            for bloco in r.iter_content(1 << 20):
                f.write(bloco)
    os.replace(destino + '.parcial', destino)
    assert zip_integro(destino), f'{os.path.basename(destino)}: o arquivo baixado não é um zip íntegro. Pare e me mande esta saída.'
    os.makedirs(os.path.dirname(no_drive), exist_ok=True)
    shutil.copyfile(destino, no_drive)
    return destino

print(f'Repositório no commit {commit[:7]}. Tiles:', {t: tuple(c) for t, c in CAIXAS.items()})

In [ ]:
# Célula 2 de 5: produtos escolhidos pela regra do plano
DIA = '2024-05-06'
api = json.load(open(f'{REPO_GIT}/analysis/rq4/referencias/emsr720_api.json', encoding='utf-8'))['results'][0]
SELECIONADOS = []
for a in api['aois']:
    for p in a['products']:
        imagens = sorted({(i['sensorName'], i['acquisitionTime']) for i in p['images']})
        # Regra do plano: produto de mapeamento de danos (GRA) da entrega principal, e todas as suas imagens de origem do dia 06/05.
        if p['type'] == 'GRA' and p.get('monitoringNumber') == 0 and imagens and all(h.startswith(DIA) for _, h in imagens) and p.get('downloadPath'):
            SELECIONADOS.append({'aoi': a['number'], 'nome': a['name'], 'produto': p['id'], 'versao': p['version']['number'],
                                 'sensor': imagens[0][0], 'imagens_utc': [h for _, h in imagens], 'url': p['downloadPath'],
                                 'camadas_na_api': [l['name'].split('/')[-1] for l in p['layers']]})
SELECIONADOS.sort(key=lambda s: s['aoi'])
assert [s['produto'] for s in SELECIONADOS] == [686, 685, 687, 682, 688], [s['produto'] for s in SELECIONADOS]
for s in SELECIONADOS:
    print(f"AOI{s['aoi']} {s['nome']:15s} produto {s['produto']} v{s['versao']}  {s['sensor']}  {', '.join(s['imagens_utc'])} UTC")
    print('    ', s['url'])
print('\nNenhum arquivo foi baixado nesta célula.')

In [ ]:
# Célula 3 de 5: baixar e extrair os pacotes
EXTRAIDO = f'{LOCAL}/extraido'
ARQUIVOS, CONTEUDO = {}, []
for s in SELECIONADOS:
    nome_zip = s['url'].rsplit('/', 1)[1]
    caminho = baixar_zip(s['url'], f'{LOCAL}/zips/{nome_zip}')
    ARQUIVOS[nome_zip] = {'url': s['url'], 'bytes': os.path.getsize(caminho), 'sha256': sha256_arquivo(caminho)}
    destino = f"{EXTRAIDO}/AOI{s['aoi']:02d}"
    os.makedirs(destino, exist_ok=True)
    with zipfile.ZipFile(caminho) as z:
        n_itens = len(z.infolist())
        for item in z.infolist():
            CONTEUDO.append(f"AOI{s['aoi']:02d}\t{item.file_size}\t{item.filename}")
            if item.is_dir() or item.filename.lower().endswith(('.tif', '.tiff', '.jp2', '.png', '.jpg')):
                continue   # as imagens não são extraídas: só as camadas vetoriais e os metadados
            z.extract(item, destino)
    print(f"AOI{s['aoi']} {nome_zip}: {os.path.getsize(caminho) / 1e6:.1f} MB, {n_itens} itens no pacote")
with open(f'{SAIDA}/emsr720_conteudo.tsv', 'w', encoding='utf-8') as f:
    f.write('aoi\tbytes\tarquivo\n' + '\n'.join(CONTEUDO) + '\n')
print('\nLista completa do conteúdo dos pacotes em emsr720_conteudo.tsv')

In [ ]:
# Célula 4 de 5: inventário das camadas e das classes
import warnings
warnings.filterwarnings('ignore')
t0 = time.time()
linhas, valores = I.inventariar([('emsr720', EXTRAIDO)], CAIXAS, base=LOCAL)
I.escrever(linhas, valores, CAIXAS, SAIDA)
print(f'{len(linhas)} camadas inventariadas em {time.time() - t0:.0f} s.\n')
for l in linhas:
    nome = f"{l.get('arquivo', '')[-70:]} :: {l.get('camada', '')}"
    areas = '  '.join(f"{t} {l.get(f'area_km2_{t}', '')!s:>8s}" for t in CAIXAS)
    print(f"{nome:105s} n {l.get('n_feicoes', '')!s:>5s}  total {l.get('area_km2', '')!s:>8s} km²  {areas}  {l.get('erro', '')}")
print('\nClasses das camadas observedEvent:')
for k, v in valores.items():
    if 'observedEvent' in k and v:
        print(' ', k.split(' | ', 2)[-1] if ' | ' in k else k)
        for col, cont in v.items():
            if col != '_linhas':
                print('     ', col, cont)

In [ ]:
# Célula 5 de 5: camadas observedEvent e registro da rodada
OBS = f'{SAIDA}/observedEvent'
os.makedirs(OBS, exist_ok=True)
copiados = []
for caminho in sorted(glob.glob(f'{EXTRAIDO}/**/*', recursive=True)):
    if os.path.isfile(caminho) and 'observedEvent' in os.path.basename(caminho):
        rel = os.path.relpath(caminho, EXTRAIDO).replace('/', '__')
        shutil.copyfile(caminho, f'{OBS}/{rel}')
        copiados.append(rel)
print(f'{len(copiados)} arquivos observedEvent copiados:', copiados)

from importlib.metadata import version
INFO = {'plano': 'prereg/plano_rq4.md, secao 2', 'commit_repositorio': commit,
        'inventario_sha256': sha256_arquivo(f'{REPO_GIT}/analysis/rq4/inventario_referencias.py'),
        'produtos': SELECIONADOS, 'arquivos_baixados': ARQUIVOS,
        'versoes': {p: version(p) for p in ('geopandas', 'pyogrio', 'shapely')},
        'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA}/rodada.json', 'w'), indent=2, ensure_ascii=False)
print('\nArquivos para me mandar (a pasta inteira, com a subpasta observedEvent):', sorted(os.listdir(SAIDA)))
print('Nenhum modelo foi rodado.')